# Declaration — ScamShield BD (NSUCEC NeuraHack 2026)

**Team:** <TEAM NAME> (<TEAM CODE>) · Anadi, Rabib, Samprity

## 1. Libraries and versions
No installation is required on Kaggle: the notebook uses the preinstalled scientific stack and prints the
exact versions it ran with in the next cell. Minimum versions: python>=3.10, numpy>=1.24, pandas>=2.0,
scikit-learn>=1.2, scipy>=1.10, PyYAML>=6.0. Developed and tested locally with python 3.14.6, numpy 2.5.1,
pandas 3.0.5, scikit-learn 1.9.1, scipy 1.18.1, PyYAML 6.0.3.

## 2. External resources
| Resource | Type | Source | License / provider |
|---|---|---|---|
| scikit-learn, numpy, pandas, scipy, PyYAML | libraries | PyPI | BSD-3 / BSD-3 / BSD-3 / BSD-3 / MIT |
| (add every pretrained model, dataset and public code here before submitting) | | | |

- **Hosted APIs:** none. The `.env` in the ZIP is empty.
- **External labelled datasets:** none.
- **Pseudo-labeling / use of unlabeled test text:** none in this version. (If added: done by code below,
  using only model predictions.)
- **Synthetic data:** none in this version.

## 3. Method and validation
- Text is kept raw and also normalized conservatively (Unicode NFC, invisible-character removal outside Bangla
  conjuncts, Cyrillic/Greek homoglyph folding inside Latin tokens, Latin lowercasing). URLs, numbers, emojis
  and Bangla marks are preserved.
- Model: character (2–5, word-boundary) + word (1–2) TF-IDF → logistic regression (balanced class weights),
  one multiclass model for Task A, one for Task C, six binary models for Task B. Task D confidence = max Task A
  probability. Rows predicted Legitimate (label 0) get techniques NONE.
- Validation: shared grouped stratified 5-fold CV (seed 42). Groups merge rows with the same template
  (digits/URLs masked) or char TF-IDF cosine ≥ 0.80, so near-duplicates never cross folds. Vectorizers are
  fit inside each training fold.
- Local proxies for the hidden subsets (not Kaggle scores): robustness = validation text corrupted with
  homoglyphs / zero-width chars / junk tokens / repeats; novel scenarios = whole scenario clusters held out;
  domain shift = all rows of a brand held out together.

## 4. Seeds and reproduction
- `SEED = 42` for python, numpy, scikit-learn (and torch if used).
- Run all cells top to bottom with the competition data attached. Runtime on Kaggle CPU: a few minutes.
- Output: `/kaggle/working/submission.csv`.

## Project source (embedded so this notebook runs without any external code)

In [ ]:
import os, sys
SRC = os.path.abspath('scamshield_src')
os.makedirs(SRC, exist_ok=True)
for d in ['configs','common','augmentation','models/classical']:
    os.makedirs(os.path.join(SRC, d), exist_ok=True)
print(SRC)

In [ ]:
%%writefile scamshield_src/configs/base.yaml
# Shared configuration for every model family.
# Paths are resolved by common/data_loader.py::resolve_paths. Order of search for the data dir:
#   1. env var SCAMSHIELD_DATA_DIR
#   2. data.dir below (relative to the repo root)
#   3. data.fallback_dirs (relative to the repo root)
#   4. any folder under /kaggle/input that contains sample_submission.csv
seed: 42

data:
  dir: data
  fallback_dirs: [".."]          # local layout: E:\neurahack\sample.csv next to this repo
  train_file: sample.csv          # labeled data (official name in the competition files)
  test_file: test.csv             # unlabeled test set; only present on Kaggle for us
  sample_submission_file: sample_submission.csv
  id_col: id
  text_col_candidates: [text, message, conversation, content]
  turn_separator: "|||"           # observed in sample.csv conversations: "S: ... ||| R: ..."

output:
  artifacts_dir: artifacts        # OOF / test probability arrays (env SCAMSHIELD_ARTIFACTS_DIR overrides)
  outputs_dir: outputs            # reports, fold manifest, submission (env SCAMSHIELD_OUTPUTS_DIR overrides)

folds:
  max_splits: 5
  near_dup_threshold: 0.80        # char TF-IDF cosine for "same template" grouping
  novel_rows_per_cluster: 3       # scenario clusters for the novel-scenario proxy
  n_repeats: 5                    # repeated grouped CV manifests (stability estimate)

submission:
  technique_threshold: 0.5
  # Official rule: non-fraudulent records use NONE. Only class 0 (Legitimate) is unambiguous.
  # Class 5 (Promotional spam) is NONE in 100% of training rows but the statement does not say it
  # is "non-fraudulent" -> left out until officially clarified.
  nonfraud_classes: [0]
  # If a fraud class is predicted but no technique passes the threshold, emit the single most
  # likely technique instead of NONE (every fraud row in training has >= 1 technique). Off by default.
  fraud_requires_technique: false

In [ ]:
%%writefile scamshield_src/configs/labels.yaml
# Label mappings from the ScamShield BD problem statement. Column order of every probability
# matrix follows the order of these lists. Do not reorder.
task_a:
  0: Legitimate
  1: MFS/OTP scam
  2: Prize/lottery scam
  3: Fake job/investment
  4: Delivery/parcel phishing
  5: Promotional spam

task_b:   # multi-label; empty set is written as NONE
  T1: Authority or organization impersonation
  T2: Credential or sensitive-information harvesting
  T3: Advance-fee or payment deception
  T4: Prize or reward baiting
  T5: Urgency, pressure, trust-building or social engineering
  T6: Task-based earning, fake job or investment manipulation

task_c:
  0: No suspicious action
  1: Disclose confidential information
  2: Transfer money or pay a fee
  3: Visit a link or website
  4: Install an application, join a group or complete a task
  5: Contact, call or reply

none_token: NONE
technique_separator: "|"

In [ ]:
%%writefile scamshield_src/common/__init__.py
"""Shared ScamShield BD infrastructure. Every model family imports from here."""
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent.parent

In [ ]:
%%writefile scamshield_src/common/seed.py
import os
import random

import numpy as np

SEED = 42


def set_seed(seed: int = SEED) -> int:
    """Seed python, numpy and (if installed) torch. Returns the seed."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
    return seed


def device_info() -> dict:
    info = {"cpu_count": os.cpu_count(), "gpu": None}
    try:
        import torch

        if torch.cuda.is_available():
            info["gpu"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except ImportError:
        pass
    return info

In [ ]:
%%writefile scamshield_src/common/labels.py
"""Label constants and technique encoding. Single source of truth for column order."""
from __future__ import annotations

import numpy as np

TASK_A_CLASSES = [0, 1, 2, 3, 4, 5]
TASK_A_NAMES = {
    0: "Legitimate",
    1: "MFS/OTP scam",
    2: "Prize/lottery scam",
    3: "Fake job/investment",
    4: "Delivery/parcel phishing",
    5: "Promotional spam",
}
TECHNIQUES = ["T1", "T2", "T3", "T4", "T5", "T6"]
TECHNIQUE_NAMES = {
    "T1": "Authority or organization impersonation",
    "T2": "Credential or sensitive-information harvesting",
    "T3": "Advance-fee or payment deception",
    "T4": "Prize or reward baiting",
    "T5": "Urgency, pressure, trust-building or social engineering",
    "T6": "Task-based earning, fake job or investment manipulation",
}
TASK_C_CLASSES = [0, 1, 2, 3, 4, 5]
TASK_C_NAMES = {
    0: "No suspicious action",
    1: "Disclose confidential information",
    2: "Transfer money or pay a fee",
    3: "Visit a link or website",
    4: "Install an application, join a group or complete a task",
    5: "Contact, call or reply",
}
NONE_TOKEN = "NONE"
SEP = "|"
N_A, N_B, N_C = len(TASK_A_CLASSES), len(TECHNIQUES), len(TASK_C_CLASSES)


class LabelError(ValueError):
    pass


def parse_techniques(value) -> list[str]:
    """'T1|T5' -> ['T1','T5']; 'NONE' -> []. Raises LabelError on unknown codes or duplicates."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        raise LabelError("missing techniques value (expected 'NONE' or T-codes)")
    s = str(value).strip()
    if s.upper() == NONE_TOKEN:
        return []
    codes = [c.strip().upper() for c in s.split(SEP)]
    bad = [c for c in codes if c not in TECHNIQUES]
    if bad:
        raise LabelError(f"unknown technique code(s) {bad} in {value!r}")
    if len(set(codes)) != len(codes):
        raise LabelError(f"duplicate technique code in {value!r}")
    return codes


def techniques_to_binary(values) -> np.ndarray:
    """Iterable of technique strings -> (n, 6) int8 matrix in T1..T6 order."""
    values = list(values)
    out = np.zeros((len(values), N_B), dtype=np.int8)
    for i, v in enumerate(values):
        for c in parse_techniques(v):
            out[i, TECHNIQUES.index(c)] = 1
    return out


def binary_to_techniques(row) -> str:
    """Binary/boolean vector in T1..T6 order -> canonical string ('T1|T5' or 'NONE')."""
    codes = [TECHNIQUES[j] for j in range(N_B) if row[j]]
    return SEP.join(codes) if codes else NONE_TOKEN

In [ ]:
%%writefile scamshield_src/common/schema.py
"""Schema checks for the official files. Columns come from the real files (inspected 26 Nov):
    sample.csv            id,text,label,techniques,action
    sample_submission.csv id,label,techniques,action,confidence
    test.csv              not available locally; expected id + one text column (auto-detected)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .labels import TASK_A_CLASSES, TASK_C_CLASSES, LabelError, parse_techniques

TRAIN_LABEL_COLS = ["label", "techniques", "action"]
SUBMISSION_COLS = ["id", "label", "techniques", "action", "confidence"]


class SchemaError(ValueError):
    pass


def find_text_col(df: pd.DataFrame, candidates, id_col="id") -> str:
    for c in candidates:
        if c in df.columns:
            return c
    obj = [c for c in df.columns if c != id_col and not pd.api.types.is_numeric_dtype(df[c])]
    if len(obj) == 1:
        return obj[0]
    raise SchemaError(
        f"cannot identify the text column. Columns: {list(df.columns)}. "
        f"Add the right name to data.text_col_candidates in configs/base.yaml."
    )


def check_ids(df: pd.DataFrame, id_col: str, name: str) -> list[str]:
    if id_col not in df.columns:
        raise SchemaError(f"{name}: missing id column {id_col!r}; columns are {list(df.columns)}")
    problems = []
    if df[id_col].isna().any():
        problems.append(f"{name}: {int(df[id_col].isna().sum())} missing ids")
    dup = df[id_col][df[id_col].duplicated(keep=False)]
    if len(dup):
        problems.append(f"{name}: duplicate ids {sorted(dup.unique().tolist())[:10]}")
    return problems


def check_train_labels(df: pd.DataFrame) -> list[str]:
    """Return a list of human-readable problems (empty = valid). Never drops rows."""
    problems = []
    missing = [c for c in TRAIN_LABEL_COLS if c not in df.columns]
    if missing:
        raise SchemaError(f"train file missing label columns {missing}; columns are {list(df.columns)}")
    for col, allowed in (("label", TASK_A_CLASSES), ("action", TASK_C_CLASSES)):
        vals = pd.to_numeric(df[col], errors="coerce")
        bad = df.loc[vals.isna() | ~vals.isin(allowed), "id"].tolist()
        if bad:
            problems.append(f"{col}: invalid or missing values at ids {bad[:10]}")
        elif not np.allclose(vals, vals.round()):
            problems.append(f"{col}: non-integer values")
    for rid, v in zip(df["id"], df["techniques"]):
        try:
            parse_techniques(v)
        except LabelError as e:
            problems.append(f"techniques at id {rid}: {e}")
    return problems

In [ ]:
%%writefile scamshield_src/common/preprocessing.py
"""Conservative, configurable text normalization.

Design rules (see docs/TEAM_CONTRACT.md):
- raw_text is never modified; normalized_text is a separate view.
- Punctuation, emojis, digits, URLs, phone masks and Bangla combining marks are kept.
- ZWJ / ZWNJ are kept when they sit between Bangla characters (they shape conjuncts);
  elsewhere they are treated as adversarial invisible characters and removed.
"""
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass

BANGLA_RE = re.compile(r"[ঀ-৿]")
LATIN_RE = re.compile(r"[A-Za-z]")

# Invisible characters that never carry meaning in this data.
_ALWAYS_INVISIBLE = "​⁠﻿­᠎‎‏‪‫‬‭‮"
_INVISIBLE_RE = re.compile(f"[{_ALWAYS_INVISIBLE}]")
_ZWJ_ZWNJ = "‌‍"

# Cyrillic / Greek letters that look like Latin letters (seen in training data, e.g. 'sаlаry').
HOMOGLYPHS = {
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ј": "j",
    "ѕ": "s", "ԁ": "d", "ɡ": "g", "һ": "h", "ӏ": "l", "ո": "n", "ս": "u", "ν": "v", "ԝ": "w",
    "А": "A", "В": "B", "Е": "E", "К": "K", "М": "M", "Н": "H", "О": "O", "Р": "P", "С": "C",
    "Т": "T", "Х": "X", "У": "Y", "І": "I", "Ј": "J", "Ѕ": "S",
    "α": "a", "ο": "o", "ρ": "p", "ε": "e", "ι": "i", "κ": "k", "τ": "t", "υ": "u",
    "Α": "A", "Β": "B", "Ε": "E", "Ζ": "Z", "Η": "H", "Ι": "I", "Κ": "K", "Μ": "M", "Ν": "N",
    "Ο": "O", "Ρ": "P", "Τ": "T", "Υ": "Y", "Χ": "X",
}
_HOMO_TABLE = str.maketrans(HOMOGLYPHS)

_REPEAT_RE = re.compile(r"([^\W\d_])\1{2,}", re.UNICODE)  # 3+ identical letters -> 2
_WS_RE = re.compile(r"[ \t\r\f\v  - 　]+")
_DEFANG = [(re.compile(r"\[\.\]"), "."), (re.compile(r"\bhxxp(s?)://", re.I), r"http\1://")]


@dataclass(frozen=True)
class NormConfig:
    unicode_form: str = "NFC"        # NFC keeps Bangla vowel signs intact; NFKC also folds fullwidth
    remove_invisible: bool = True
    homoglyphs: bool = True
    lowercase: bool = True           # Latin only; Bangla has no case
    squeeze_repeats: bool = False    # optional view: 'PIIIN' -> 'PIIN'
    refang_urls: bool = False        # optional view: 'hxxps://x[.]top' -> 'https://x.top'
    empty_token: str = ""            # what a fully-empty record becomes (e.g. '<empty>' for tokenizers)


DEFAULT = NormConfig()
AGGRESSIVE = NormConfig(unicode_form="NFKC", squeeze_repeats=True, refang_urls=True)


def _strip_zw_outside_bangla(text: str) -> str:
    if not any(c in text for c in _ZWJ_ZWNJ):
        return text
    out = []
    for i, ch in enumerate(text):
        if ch in _ZWJ_ZWNJ:
            prev_b = i > 0 and BANGLA_RE.match(text[i - 1])
            next_b = i + 1 < len(text) and BANGLA_RE.match(text[i + 1])
            if prev_b and next_b:
                out.append(ch)
            continue
        out.append(ch)
    return "".join(out)


def _fold_homoglyphs(text: str) -> str:
    # Only fold inside tokens that also contain real Latin letters, so genuine Cyrillic/Greek words
    # (not expected here, but possible) are left alone.
    def fix(m: re.Match) -> str:
        tok = m.group(0)
        return tok.translate(_HOMO_TABLE) if LATIN_RE.search(tok) else tok

    return re.sub(r"\S+", fix, text)


def normalize(text, cfg: NormConfig = DEFAULT) -> str:
    if text is None or (isinstance(text, float) and text != text):
        return cfg.empty_token
    s = unicodedata.normalize(cfg.unicode_form, str(text))
    if cfg.remove_invisible:
        s = _INVISIBLE_RE.sub("", s)
        s = _strip_zw_outside_bangla(s)
    if cfg.homoglyphs:
        s = _fold_homoglyphs(s)
    if cfg.refang_urls:
        for pat, rep in _DEFANG:
            s = pat.sub(rep, s)
    if cfg.lowercase:
        s = s.lower()  # str.lower leaves Bangla untouched
    if cfg.squeeze_repeats:
        s = _REPEAT_RE.sub(r"\1\1", s)
    s = _WS_RE.sub(" ", s)
    s = re.sub(r"\n+", " \n ", s).strip()
    s = re.sub(r" {2,}", " ", s)
    return s if s else cfg.empty_token


def script_profile(text: str) -> dict:
    """Script shares. NOTE: 'latin' cannot tell English from Banglish; only the script is detected."""
    s = "" if text is None else str(text)
    n_b = len(BANGLA_RE.findall(s))
    n_l = len(LATIN_RE.findall(s))
    tot = n_b + n_l
    if tot == 0:
        script = "none"
    elif n_b / tot >= 0.9:
        script = "bangla"
    elif n_l / tot >= 0.9:
        script = "latin"
    else:
        script = "mixed"
    non_latin_letters = sum(1 for c in s if c.isalpha() and not LATIN_RE.match(c) and not BANGLA_RE.match(c))
    invis = len(_INVISIBLE_RE.findall(s)) + sum(s.count(c) for c in _ZWJ_ZWNJ)
    return {
        "script": script,
        "bangla_chars": n_b,
        "latin_chars": n_l,
        "other_letters": non_latin_letters,      # homoglyph signal (Cyrillic/Greek)
        "invisible_chars": invis,
        "has_url": bool(re.search(r"(https?|hxxps?)://|\[\.\]|\bwww\.", s, re.I)),
    }


def is_noise_only(text: str) -> bool:
    """Heuristic flag: no Bangla, no digits/URL, and no Latin token with a vowel pattern of a real word.
    Used only for EDA/diagnostics, never to drop rows."""
    s = normalize(text)
    if not s or BANGLA_RE.search(s) or re.search(r"\d|://|\[\.\]", s):
        return False
    toks = re.findall(r"[a-z]+", s)
    if not toks:
        return True
    return all(is_junk_token(t) for t in toks)


def is_junk_token(tok: str) -> bool:
    """Keyboard-mash token: a run of 5+ consonants, or 4+ letters with no vowel at all."""
    t = tok.lower()
    return bool(re.search(r"[bcdfghjklmnpqrstvwxz]{5,}", t)) or (len(t) >= 4 and not re.search(r"[aeiouy]", t))

In [ ]:
%%writefile scamshield_src/common/data_loader.py
"""One loader for every model family.

    from common.data_loader import load_config, load_train, load_test, get_targets
    cfg = load_config()
    train = load_train(cfg)          # canonical DataFrame, original order, original ids
    yA, yB, yC = get_targets(train)  # (n,), (n,6), (n,)

Canonical columns: row_idx, id, raw_text, normalized_text, turns, n_turns, speakers,
is_conversation, script, (train only) label, action, techniques, T1..T6.
"""
from __future__ import annotations

import hashlib
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from . import REPO_ROOT
from .labels import TECHNIQUES, binary_to_techniques, techniques_to_binary
from .preprocessing import DEFAULT, NormConfig, normalize, script_profile
from .schema import SchemaError, check_ids, check_train_labels, find_text_col

# Leading junk (e.g. an injected emoji) before the speaker tag is kept as content.
_SPEAKER_RE = re.compile(r"^([^\w]*?)\s*([A-Za-z]{1,12})\s*:\s*(.*)$", re.S)


def load_config(path: str | Path | None = None) -> dict:
    path = Path(path) if path else REPO_ROOT / "configs" / "base.yaml"
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def _kaggle_dirs():
    root = Path("/kaggle/input")
    if root.exists():
        for p in sorted(root.rglob("sample_submission.csv")):
            yield p.parent


def resolve_data_dir(cfg: dict) -> Path:
    d = cfg["data"]
    tried = []
    cands = []
    if os.getenv("SCAMSHIELD_DATA_DIR"):
        cands.append(Path(os.environ["SCAMSHIELD_DATA_DIR"]))
    cands.append(REPO_ROOT / d["dir"])
    cands += [REPO_ROOT / x for x in d.get("fallback_dirs", [])]
    cands += list(_kaggle_dirs())
    for c in cands:
        tried.append(str(c))
        if (c / d["train_file"]).exists() or (c / d["sample_submission_file"]).exists():
            return c.resolve()
    raise FileNotFoundError(
        f"No data dir found containing {d['train_file']} or {d['sample_submission_file']}. "
        f"Tried: {tried}. Set SCAMSHIELD_DATA_DIR."
    )


def output_dirs(cfg: dict) -> tuple[Path, Path]:
    o = cfg["output"]
    art = Path(os.getenv("SCAMSHIELD_ARTIFACTS_DIR", REPO_ROOT / o["artifacts_dir"]))
    out = Path(os.getenv("SCAMSHIELD_OUTPUTS_DIR", REPO_ROOT / o["outputs_dir"]))
    art.mkdir(parents=True, exist_ok=True)
    out.mkdir(parents=True, exist_ok=True)
    return art, out


def file_fingerprint(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def split_turns(text: str, sep: str = "|||") -> list[tuple[str | None, str]]:
    """'S: hi ||| R: what?' -> [('S','hi'), ('R','what?')]. Single messages -> [(None, text)]."""
    if text is None or (isinstance(text, float) and text != text):
        return [(None, "")]
    parts = [p.strip() for p in str(text).split(sep)] if sep in str(text) else [str(text).strip()]
    turns = []
    for p in parts:
        m = _SPEAKER_RE.match(p)
        # Only treat a prefix as a speaker when the message is a multi-turn conversation;
        # single messages like 'bKash Helpline: ...' or 'Boss: ...' keep their prefix as content.
        if m and len(parts) > 1 and len(m.group(2)) <= 2:
            content = (m.group(1).strip() + " " + m.group(3).strip()).strip()
            turns.append((m.group(2).upper(), content))
        else:
            turns.append((None, p))
    return turns


def _read_csv(path: Path) -> pd.DataFrame:
    # keep_default_na=False: a message like "NA" or "None" must stay text, never become NaN.
    return pd.read_csv(path, keep_default_na=False, na_values=[""], dtype={"text": str}, encoding="utf-8")


def _canonicalize(df: pd.DataFrame, cfg: dict, text_col: str, norm: NormConfig) -> pd.DataFrame:
    sep = cfg["data"].get("turn_separator", "|||")
    raw = df[text_col].astype(object).where(df[text_col].notna(), "")
    out = pd.DataFrame({"row_idx": np.arange(len(df)), "id": df[cfg["data"]["id_col"]].to_numpy()})
    out["raw_text"] = raw.astype(str).to_numpy()
    out["normalized_text"] = [normalize(t, norm) for t in out["raw_text"]]
    out["turns"] = [split_turns(t, sep) for t in out["raw_text"]]
    out["n_turns"] = out["turns"].map(len)
    out["speakers"] = out["turns"].map(lambda ts: "|".join(s or "-" for s, _ in ts))
    out["is_conversation"] = out["n_turns"] > 1
    out["script"] = out["raw_text"].map(lambda t: script_profile(t)["script"])
    return out


def load_train(cfg: dict | None = None, norm: NormConfig = DEFAULT, strict: bool = True) -> pd.DataFrame:
    cfg = cfg or load_config()
    path = resolve_data_dir(cfg) / cfg["data"]["train_file"]
    df = _read_csv(path)
    id_col = cfg["data"]["id_col"]
    text_col = find_text_col(df, cfg["data"]["text_col_candidates"], id_col)
    problems = check_ids(df, id_col, "train") + check_train_labels(df)
    if problems and strict:
        raise SchemaError("train file failed validation:\n  " + "\n  ".join(problems))
    out = _canonicalize(df, cfg, text_col, norm)
    out["label"] = df["label"].astype(int).to_numpy()
    out["action"] = df["action"].astype(int).to_numpy()
    tb = techniques_to_binary(df["techniques"])
    out["techniques"] = [binary_to_techniques(r) for r in tb]  # canonical order T1..T6
    for j, t in enumerate(TECHNIQUES):
        out[t] = tb[:, j]
    out.attrs.update(source=str(path), text_col=text_col, fingerprint=file_fingerprint(path), problems=problems)
    return out


def load_test(cfg: dict | None = None, norm: NormConfig = DEFAULT) -> pd.DataFrame:
    """Loads test.csv. Raises FileNotFoundError when it is not present (expected locally)."""
    cfg = cfg or load_config()
    path = resolve_data_dir(cfg) / cfg["data"]["test_file"]
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. test.csv is only available in the Kaggle environment.")
    df = _read_csv(path)
    id_col = cfg["data"]["id_col"]
    text_col = find_text_col(df, cfg["data"]["text_col_candidates"], id_col)
    problems = check_ids(df, id_col, "test")
    if problems:
        raise SchemaError("test file failed validation:\n  " + "\n  ".join(problems))
    out = _canonicalize(df, cfg, text_col, norm)
    out.attrs.update(source=str(path), text_col=text_col, fingerprint=file_fingerprint(path),
                     extra_columns=[c for c in df.columns if c not in (id_col, text_col)])
    return out


def load_sample_submission(cfg: dict | None = None) -> pd.DataFrame:
    cfg = cfg or load_config()
    path = resolve_data_dir(cfg) / cfg["data"]["sample_submission_file"]
    df = pd.read_csv(path, keep_default_na=False)
    problems = check_ids(df, cfg["data"]["id_col"], "sample_submission")
    if problems:
        raise SchemaError("; ".join(problems))
    df.attrs["fingerprint"] = file_fingerprint(path)
    return df


def test_available(cfg: dict | None = None) -> bool:
    cfg = cfg or load_config()
    try:
        return (resolve_data_dir(cfg) / cfg["data"]["test_file"]).exists()
    except FileNotFoundError:
        return False


def get_targets(train: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    return (train["label"].to_numpy(int), train[TECHNIQUES].to_numpy(np.int8), train["action"].to_numpy(int))

In [ ]:
%%writefile scamshield_src/common/folds.py
"""Shared grouped CV. Everyone reads folds/folds.csv (id,fold); nobody makes their own splits.

Groups: rows are merged into one group when they share a template key (digits/URLs/case masked)
or their char TF-IDF cosine >= near_dup_threshold. StratifiedGroupKFold (seed 42) then keeps each
group inside one fold, stratified on Task A.

Extra manifests for proxy scores (same id,fold format):
  folds_repeats.csv  id,repeat,fold  - 5 grouped re-splits, for mean +/- std
  folds_novel.csv    scenario clusters (complete-linkage, ~3 rows each) held out whole -> novel-scenario proxy
  folds_domain.csv   leave-brand-out (all rows mentioning the same brand in one fold) -> domain proxy
"""
from __future__ import annotations

import hashlib
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedGroupKFold

from . import REPO_ROOT
from .labels import TASK_A_CLASSES, TASK_C_CLASSES, TECHNIQUES

FOLDS_DIR = Path(os.getenv("SCAMSHIELD_FOLDS_DIR", REPO_ROOT / "folds"))


def _fdir(fold_dir) -> Path:
    """Resolve at call time so FOLDS_DIR / SCAMSHIELD_FOLDS_DIR can be changed after import."""
    if fold_dir is not None:
        return Path(fold_dir)
    return Path(os.getenv("SCAMSHIELD_FOLDS_DIR", FOLDS_DIR))

# Brands/operators seen in Bangladesh SMS traffic. Domain knowledge, not derived from test data.
BRANDS = {
    "bkash": r"b\s*kash|bksah|বিকাশ", "nagad": r"nagad|নগদ", "rocket": r"\brocket\b|dbbl|dutch.?bangla",
    "upay": r"\bupay\b", "daraz": r"daraz|দারাজ", "grameenphone": r"grameenphone|\bgp\b|mygp",
    "robi": r"\brobi\b|রবি", "banglalink": r"banglalink", "airtel": r"airtel", "teletalk": r"teletalk",
    "pathao": r"pathao|পাঠাও", "sundarban": r"sundarban", "sa_paribahan": r"sa paribahan|\bsap\b",
    "redx": r"\bredx\b", "steadfast": r"steadfast", "dhl": r"\bdhl\b", "fedex": r"fedex",
    "city_bank": r"city bank", "brac": r"brac", "titas": r"titas", "breb": r"breb|desco|dpdc",
    "unilever": r"unilever", "aarong": r"aarong", "shwapno": r"shwapno", "bdjobs": r"bdjobs",
    "telegram": r"telegram|t\[?\.\]?me", "whatsapp": r"whats\s*app|হোয়াটসঅ্যাপ", "facebook": r"facebook|facccebook",
    "youtube": r"youtube", "pizza_hut": r"pizza hut", "foodpanda": r"foodpanda",
}
_BRAND_RES = {k: re.compile(v, re.I) for k, v in BRANDS.items()}


def template_key(text: str) -> str:
    s = text.lower()
    s = re.sub(r"(https?|hxxps?)://\S+|\S+\[\.\]\S+", " <url> ", s)
    s = re.sub(r"[0-9০-৯]+", "0", s)
    s = re.sub(r"[^\w<> ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def detect_brands(text: str) -> list[str]:
    return [k for k, r in _BRAND_RES.items() if r.search(text)]


class _UF:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[max(ra, rb)] = min(ra, rb)


def similarity_matrix(texts) -> np.ndarray:
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1, sublinear_tf=True)
    X = vec.fit_transform(texts)
    S = (X @ X.T).toarray()
    np.fill_diagonal(S, 0.0)
    return S


def build_groups(texts, threshold: float, S: np.ndarray | None = None) -> np.ndarray:
    texts = list(texts)
    n = len(texts)
    uf = _UF(n)
    first = {}
    for i, t in enumerate(texts):
        k = template_key(t)
        if k in first:
            uf.union(i, first[k])
        else:
            first[k] = i
    S = similarity_matrix(texts) if S is None else S
    ii, jj = np.where(np.triu(S, 1) >= threshold)
    for a, b in zip(ii, jj):
        uf.union(int(a), int(b))
    roots = [uf.find(i) for i in range(n)]
    _, gid = np.unique(roots, return_inverse=True)
    return gid


def scenario_clusters(S: np.ndarray, rows_per_cluster: int) -> np.ndarray:
    """Complete-linkage clusters on cosine distance. Groups related scenarios even when no pair is a
    near-duplicate (the case for the 86-row sample)."""
    n = S.shape[0]
    k = max(2, n // max(1, rows_per_cluster))
    D = np.clip(1.0 - S, 0.0, None)
    np.fill_diagonal(D, 0.0)
    return AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="complete").fit_predict(D)


def choose_n_splits(y, groups, max_splits: int) -> int:
    y = np.asarray(y)
    min_class_groups = min(len(np.unique(groups[y == c])) for c in np.unique(y))
    return int(max(2, min(max_splits, min_class_groups, len(np.unique(groups)))))


def grouped_folds(y, groups, n_splits: int, seed: int) -> np.ndarray:
    fold = np.full(len(y), -1, dtype=int)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, va) in enumerate(sgkf.split(np.zeros(len(y)), y, groups)):
        fold[va] = k
    assert (fold >= 0).all()
    return fold


def manifest_fingerprint(df: pd.DataFrame) -> str:
    return hashlib.sha256(df.to_csv(index=False, lineterminator="\n").encode()).hexdigest()[:16]


def coverage_report(train: pd.DataFrame, fold: np.ndarray) -> dict:
    rep = {}
    for k in sorted(np.unique(fold)):
        m = fold == k
        sub = train[m]
        rep[int(k)] = {
            "n": int(m.sum()),
            "task_a": {int(c): int((sub.label == c).sum()) for c in TASK_A_CLASSES},
            "task_c": {int(c): int((sub.action == c).sum()) for c in TASK_C_CLASSES},
            "techniques": {t: int(sub[t].sum()) for t in TECHNIQUES},
        }
    return rep


def leakage_report(train: pd.DataFrame, fold: np.ndarray, S: np.ndarray, warn_at: float = 0.6) -> list[dict]:
    """Cross-fold pairs that are still fairly similar (below the grouping threshold)."""
    ii, jj = np.where(np.triu(S, 1) >= warn_at)
    out = []
    for a, b in zip(ii, jj):
        if fold[a] != fold[b]:
            out.append({"id_a": train.id.iloc[a].item(), "id_b": train.id.iloc[b].item(),
                        "cosine": round(float(S[a, b]), 3)})
    return sorted(out, key=lambda d: -d["cosine"])


def make_all_folds(train: pd.DataFrame, cfg: dict, out_dir: Path | None = None) -> dict:
    fc, seed = cfg["folds"], int(cfg["seed"])
    out_dir = _fdir(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    texts = train["normalized_text"].tolist()
    y = train["label"].to_numpy()
    S = similarity_matrix(texts)

    groups = build_groups(texts, fc["near_dup_threshold"], S)
    k = choose_n_splits(y, groups, fc["max_splits"])
    fold = grouped_folds(y, groups, k, seed)
    main = pd.DataFrame({"id": train.id.to_numpy(), "fold": fold})
    main.to_csv(out_dir / "folds.csv", index=False, lineterminator="\n")

    reps = []
    for r in range(int(fc["n_repeats"])):
        reps.append(pd.DataFrame({"id": train.id.to_numpy(), "repeat": r,
                                  "fold": grouped_folds(y, groups, k, seed + r)}))
    reps = pd.concat(reps, ignore_index=True)
    reps.to_csv(out_dir / "folds_repeats.csv", index=False, lineterminator="\n")

    g_novel = scenario_clusters(S, int(fc["novel_rows_per_cluster"]))
    k_novel = choose_n_splits(y, g_novel, fc["max_splits"])
    novel = pd.DataFrame({"id": train.id.to_numpy(), "fold": grouped_folds(y, g_novel, k_novel, seed)})
    novel.to_csv(out_dir / "folds_novel.csv", index=False, lineterminator="\n")

    brands = [detect_brands(t) for t in train["raw_text"]]
    g_dom = np.array([("brand:" + b[0]) if b else f"grp:{g}" for b, g in zip(brands, groups)])
    _, g_dom_i = np.unique(g_dom, return_inverse=True)
    k_dom = choose_n_splits(y, g_dom_i, fc["max_splits"])
    domain = pd.DataFrame({"id": train.id.to_numpy(), "fold": grouped_folds(y, g_dom_i, k_dom, seed)})
    domain.to_csv(out_dir / "folds_domain.csv", index=False, lineterminator="\n")

    meta = {
        "seed": seed,
        "dataset_fingerprint": train.attrs.get("fingerprint"),
        "n_rows": int(len(train)),
        "main": {"n_splits": k, "n_groups": int(groups.max() + 1), "threshold": fc["near_dup_threshold"],
                 "fingerprint": manifest_fingerprint(main),
                 "multi_row_groups": int((np.bincount(groups) > 1).sum()),
                 "coverage": coverage_report(train, fold),
                 "leakage_risk_pairs": leakage_report(train, fold, S)[:25]},
        "repeats": {"n_repeats": int(fc["n_repeats"]), "fingerprint": manifest_fingerprint(reps)},
        "novel": {"n_splits": k_novel, "n_groups": int(g_novel.max() + 1), "rows_per_cluster": fc["novel_rows_per_cluster"],
                  "cluster_sizes": np.bincount(g_novel).tolist(),
                  "fingerprint": manifest_fingerprint(novel)},
        "domain": {"n_splits": k_dom, "n_brand_rows": int(sum(bool(b) for b in brands)),
                   "brand_counts": pd.Series([b[0] for b in brands if b]).value_counts().to_dict(),
                   "fingerprint": manifest_fingerprint(domain)},
    }
    with open(out_dir / "folds_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    return meta


def load_folds(train: pd.DataFrame, name: str = "folds", fold_dir: Path | None = None) -> np.ndarray:
    """Return fold numbers aligned to train row order, verified by id."""
    m = pd.read_csv(_fdir(fold_dir) / f"{name}.csv")
    if "repeat" in m.columns:
        raise ValueError("use load_repeat_folds for folds_repeats.csv")
    return _align(train, m)


def load_repeat_folds(train: pd.DataFrame, fold_dir: Path | None = None) -> list[np.ndarray]:
    m = pd.read_csv(_fdir(fold_dir) / "folds_repeats.csv")
    return [_align(train, g[["id", "fold"]]) for _, g in m.groupby("repeat", sort=True)]


def _align(train: pd.DataFrame, m: pd.DataFrame) -> np.ndarray:
    if set(m.id) != set(train.id) or len(m) != len(train):
        raise ValueError("fold manifest ids do not match the training ids; regenerate with scripts/create_folds.py")
    return train[["id"]].merge(m, on="id", how="left", validate="one_to_one")["fold"].to_numpy(int)


def fold_fingerprint(name: str = "folds", fold_dir: Path | None = None) -> str:
    return manifest_fingerprint(pd.read_csv(_fdir(fold_dir) / f"{name}.csv"))

In [ ]:
%%writefile scamshield_src/common/metrics.py
"""Official-style metrics and the LOCAL composite proxy.

OFFICIAL weights (problem statement):
  S = 0.40 F_A + 0.15 F_B + 0.10 F_C + 0.10 F_domain + 0.10 F_robust + 0.05 Calibration + 0.10 F_novel

F_A, F_B, F_C and Calibration can be computed locally on OOF predictions with the same definitions.
F_domain / F_robust / F_novel are scored by the organizers on HIDDEN subsets. Locally we only have
PROXIES (see common/stress.py). A composite that uses proxies is a LOCAL PROXY, never a Kaggle score.

Assumptions to verify against the official evaluation page:
  - Task B: per-technique binary F1 with zero_division=0, averaged over T1..T6.
  - Calibration: 1 - mean((confidence - 1[pred_A == y_A])^2).
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

from .labels import TASK_A_CLASSES, TASK_C_CLASSES, TECHNIQUES

WEIGHTS = {"F_A": 0.40, "F_B": 0.15, "F_C": 0.10, "F_domain": 0.10, "F_robust": 0.10,
           "Calibration": 0.05, "F_novel": 0.10}


def macro_f1(y_true, y_pred, labels=TASK_A_CLASSES) -> float:
    return float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0))


def task_b_f1(Y_true, Y_pred) -> float:
    Y_true, Y_pred = np.asarray(Y_true), np.asarray(Y_pred)
    return float(np.mean([f1_score(Y_true[:, j], Y_pred[:, j], zero_division=0) for j in range(Y_true.shape[1])]))


def brier(confidence, correct) -> float:
    c = np.asarray(confidence, float)
    return float(np.mean((c - np.asarray(correct, float)) ** 2))


def calibration_score(confidence, correct) -> float:
    return 1.0 - brier(confidence, correct)


def per_class(y_true, y_pred, labels) -> dict:
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    return {str(l): {"precision": round(float(p[i]), 4), "recall": round(float(r[i]), 4),
                     "f1": round(float(f[i]), 4), "support": int(s[i])} for i, l in enumerate(labels)}


def per_technique(Y_true, Y_pred) -> dict:
    out = {}
    for j, t in enumerate(TECHNIQUES):
        p, r, f, _ = precision_recall_fscore_support(Y_true[:, j], Y_pred[:, j], average="binary", zero_division=0)
        out[t] = {"precision": round(float(p), 4), "recall": round(float(r), 4), "f1": round(float(f), 4),
                  "support": int(np.asarray(Y_true)[:, j].sum())}
    return out


def evaluate(yA, yB, yC, pA, pB, pC, confidence=None, technique_threshold=0.5) -> dict:
    """Full evaluation from probability matrices (contract shapes, see prediction_io.py)."""
    predA, predC = pA.argmax(1), pC.argmax(1)
    predB = (pB >= technique_threshold).astype(int)
    conf = pA.max(1) if confidence is None else np.asarray(confidence)
    correct = (predA == yA).astype(int)
    return {
        "F_A": round(macro_f1(yA, predA), 4),
        "F_B": round(task_b_f1(yB, predB), 4),
        "F_C": round(macro_f1(yC, predC, TASK_C_CLASSES), 4),
        "Brier": round(brier(conf, correct), 4),
        "Calibration": round(calibration_score(conf, correct), 4),
        "accuracy_A": round(float(correct.mean()), 4),
        "per_class_A": per_class(yA, predA, TASK_A_CLASSES),
        "per_technique_B": per_technique(np.asarray(yB), predB),
        "per_action_C": per_class(yC, predC, TASK_C_CLASSES),
        "confusion_A": confusion_matrix(yA, predA, labels=TASK_A_CLASSES).tolist(),
        "confusion_C": confusion_matrix(yC, predC, labels=TASK_C_CLASSES).tolist(),
    }


def composite_proxy(scores: dict) -> dict:
    """Weighted composite using whatever components are present. Missing hidden-subset proxies are
    replaced by F_A and listed under 'imputed' so nobody mistakes the number for a measured one."""
    used, imputed = {}, []
    for k in WEIGHTS:
        if scores.get(k) is not None:
            used[k] = float(scores[k])
        else:
            used[k] = float(scores["F_A"])
            imputed.append(k)
    value = sum(WEIGHTS[k] * used[k] for k in WEIGHTS)
    return {"composite_proxy": round(value, 4), "components": {k: round(v, 4) for k, v in used.items()},
            "imputed_with_F_A": imputed,
            "note": "LOCAL PROXY. F_domain/F_robust/F_novel come from local stress tests, not hidden subsets."}


def save_json(obj: dict, path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))

In [ ]:
%%writefile scamshield_src/common/stress.py
"""Shared CV runner + local stress tests (proxies for the hidden F_domain / F_robust / F_novel subsets).

A model family plugs in with ONE function:

    def fit_predict(train_df, valid_df) -> {"A": (n_valid,6), "B": (n_valid,6), "C": (n_valid,6)}

train_df / valid_df are slices of the canonical DataFrame from common.data_loader.load_train
(use raw_text or normalized_text as you like; labels are in label / T1..T6 / action).

Proxy definitions (LOCAL, approximate):
  F_robust_proxy  Task A macro F1 when VALIDATION text is corrupted by augmentation.corrupt (main folds)
  F_novel_proxy   Task A macro F1 on folds_novel.csv (looser grouping, whole scenarios held out)
  F_domain_proxy  Task A macro F1 on folds_domain.csv (all rows of a brand held out together)
"""
from __future__ import annotations

import time

import numpy as np
import pandas as pd

from augmentation.corrupt import corrupt
from .data_loader import get_targets
from .folds import load_folds, load_repeat_folds
from .labels import N_A, N_B, N_C
from .metrics import evaluate, macro_f1
from .preprocessing import normalize


def corrupt_df(df: pd.DataFrame, seed: int = 42, strength: float = 0.15) -> pd.DataFrame:
    out = df.copy()
    out["raw_text"] = [corrupt(t, seed, strength) for t in df["raw_text"]]
    out["normalized_text"] = [normalize(t) for t in out["raw_text"]]
    return out


def run_cv(fit_predict, train: pd.DataFrame, fold: np.ndarray, valid_transform=None) -> dict:
    oof = {"A": np.zeros((len(train), N_A)), "B": np.zeros((len(train), N_B)), "C": np.zeros((len(train), N_C))}
    for k in sorted(np.unique(fold)):
        tr, va = train[fold != k], train[fold == k]
        if valid_transform is not None:
            va = valid_transform(va)
        pred = fit_predict(tr, va)
        for t in oof:
            oof[t][fold == k] = pred[t]
    return oof


def slice_scores(train: pd.DataFrame, predA: np.ndarray) -> dict:
    yA = train["label"].to_numpy()
    out = {}
    for name, mask in {
        **{f"script={s}": (train.script == s).to_numpy() for s in sorted(train.script.unique())},
        "conversation": train.is_conversation.to_numpy(),
        "single_message": (~train.is_conversation).to_numpy(),
    }.items():
        if mask.sum() >= 3:
            out[name] = {"n": int(mask.sum()), "accuracy_A": round(float((predA[mask] == yA[mask]).mean()), 4),
                         "macro_f1_A_present_classes": round(macro_f1(yA[mask], predA[mask],
                                                                      sorted(np.unique(yA[mask]))), 4)}
    return out


def run_stress(fit_predict, train: pd.DataFrame, seed: int = 42, repeats: bool = True) -> tuple[dict, dict]:
    """Returns (report, oof_main). report holds official-style OOF metrics + proxy scores."""
    t0 = time.time()
    yA, yB, yC = get_targets(train)
    fold = load_folds(train, "folds")
    oof = run_cv(fit_predict, train, fold)
    report = {"main": evaluate(yA, yB, yC, oof["A"], oof["B"], oof["C"])}

    rob = run_cv(fit_predict, train, fold, valid_transform=lambda d: corrupt_df(d, seed))
    report["F_robust_proxy"] = round(macro_f1(yA, rob["A"].argmax(1)), 4)

    nov = run_cv(fit_predict, train, load_folds(train, "folds_novel"))
    report["F_novel_proxy"] = round(macro_f1(yA, nov["A"].argmax(1)), 4)

    dom = run_cv(fit_predict, train, load_folds(train, "folds_domain"))
    report["F_domain_proxy"] = round(macro_f1(yA, dom["A"].argmax(1)), 4)

    report["slices_A"] = slice_scores(train, oof["A"].argmax(1))

    if repeats:
        fa = [macro_f1(yA, run_cv(fit_predict, train, f)["A"].argmax(1)) for f in load_repeat_folds(train)]
        report["repeated_F_A"] = {"mean": round(float(np.mean(fa)), 4), "std": round(float(np.std(fa)), 4),
                                  "values": [round(x, 4) for x in fa]}
    report["runtime_sec"] = round(time.time() - t0, 1)
    return report, oof

In [ ]:
%%writefile scamshield_src/common/prediction_io.py
"""Mandatory prediction contract for every model family.

artifacts/<model_name>/
    oof_A_<model>.npy  oof_B_<model>.npy  oof_C_<model>.npy     (n_train, 6), train row order
    test_A_<model>.npy test_B_<model>.npy test_C_<model>.npy    (n_test, 6),  test row order
    oof_ids.npy  test_ids.npy  metadata.json

A and C rows are categorical distributions (sum to 1). B columns are independent probabilities (T1..T6).
Test files are optional locally (test.csv only exists on Kaggle).
"""
from __future__ import annotations

import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from . import REPO_ROOT
from .labels import N_A, N_B, N_C, TASK_A_CLASSES, TASK_C_CLASSES, TECHNIQUES

SHAPES = {"A": N_A, "B": N_B, "C": N_C}
COLUMN_ORDER = {"A": TASK_A_CLASSES, "B": TECHNIQUES, "C": TASK_C_CLASSES}


class ContractError(ValueError):
    pass


def git_commit() -> str:
    try:
        sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, capture_output=True,
                             text=True, timeout=5).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_ROOT, capture_output=True,
                               text=True, timeout=5).stdout.strip()
        return (sha or "no-commit") + ("+dirty" if dirty else "")
    except Exception:
        return "unknown"


def validate_probs(task: str, P, n_rows: int | None = None, name: str = "") -> None:
    P = np.asarray(P)
    where = f"{name} task {task}"
    if P.ndim != 2 or P.shape[1] != SHAPES[task]:
        raise ContractError(f"{where}: expected shape (n, {SHAPES[task]}), got {P.shape}")
    if n_rows is not None and P.shape[0] != n_rows:
        raise ContractError(f"{where}: expected {n_rows} rows, got {P.shape[0]}")
    if not np.isfinite(P).all():
        raise ContractError(f"{where}: contains NaN/inf")
    if (P < -1e-6).any() or (P > 1 + 1e-6).any():
        raise ContractError(f"{where}: values outside [0, 1]")
    if task in ("A", "C"):
        s = P.sum(1)
        if not np.allclose(s, 1.0, atol=1e-3):
            raise ContractError(f"{where}: rows must sum to 1 (min {s.min():.4f}, max {s.max():.4f})")


def save_predictions(model_name: str, out_root: Path, oof: dict, oof_ids, metadata: dict,
                     test: dict | None = None, test_ids=None) -> Path:
    d = Path(out_root) / model_name
    d.mkdir(parents=True, exist_ok=True)
    oof_ids = np.asarray(oof_ids)
    for t in "ABC":
        validate_probs(t, oof[t], len(oof_ids), f"oof {model_name}")
        np.save(d / f"oof_{t}_{model_name}.npy", np.asarray(oof[t], dtype=np.float32))
    np.save(d / "oof_ids.npy", oof_ids)
    if test is not None:
        test_ids = np.asarray(test_ids)
        for t in "ABC":
            validate_probs(t, test[t], len(test_ids), f"test {model_name}")
            np.save(d / f"test_{t}_{model_name}.npy", np.asarray(test[t], dtype=np.float32))
        np.save(d / "test_ids.npy", test_ids)
    meta = {
        "model_name": model_name,
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "git_commit": git_commit(),
        "column_order": {k: [str(c) for c in v] for k, v in COLUMN_ORDER.items()},
        "has_test": test is not None,
        **metadata,
    }
    for req in ("config", "seed", "dataset_fingerprint", "fold_fingerprint", "metrics"):
        if req not in meta:
            raise ContractError(f"metadata missing required key {req!r}")
    with open(d / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=str)
    return d


def load_predictions(model_dir: Path, split: str = "oof", expected_ids=None) -> dict:
    """Load one model's matrices. If expected_ids is given, rows are re-ordered to match it by id
    (never by position) and a mismatch in the id set raises ContractError."""
    d = Path(model_dir)
    name = json.loads((d / "metadata.json").read_text(encoding="utf-8"))["model_name"]
    ids = np.load(d / f"{split}_ids.npy", allow_pickle=False)
    if len(np.unique(ids)) != len(ids):
        raise ContractError(f"{name}: duplicate ids in {split}_ids.npy")
    out = {t: np.load(d / f"{split}_{t}_{name}.npy") for t in "ABC"}
    for t in "ABC":
        validate_probs(t, out[t], len(ids), f"{split} {name}")
    if expected_ids is not None:
        expected_ids = np.asarray(expected_ids)
        if len(expected_ids) != len(ids) or set(expected_ids.tolist()) != set(ids.tolist()):
            raise ContractError(f"{name}: {split} ids do not match the reference ids")
        pos = {v: i for i, v in enumerate(ids.tolist())}
        order = np.array([pos[v] for v in expected_ids.tolist()])
        out = {t: out[t][order] for t in out}
        ids = expected_ids
    out["ids"] = ids
    return out


def check_metadata_compatible(metas: list[dict]) -> None:
    """Models can only be ensembled if they used the same data and folds."""
    for key in ("dataset_fingerprint", "fold_fingerprint"):
        vals = {m.get(key) for m in metas}
        if len(vals) > 1:
            raise ContractError(f"incompatible {key} across models: {vals}")

In [ ]:
%%writefile scamshield_src/common/submission.py
"""Builds and validates submission.csv: id,label,techniques,action,confidence.

Predictions only come from probability matrices produced by model code. Nothing here edits a
prediction by hand; the only post-processing is the official consistency rule
(non-fraudulent -> NONE) and optional, config-controlled rules documented in configs/base.yaml.
"""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

from .labels import NONE_TOKEN, TECHNIQUES, binary_to_techniques
from .prediction_io import validate_probs
from .schema import SUBMISSION_COLS

_TECH_RE = re.compile(r"^(NONE|T[1-6](\|T[1-6])*)$")


def build_submission(test_ids, pA, pB, pC, confidence=None, technique_threshold=0.5,
                     nonfraud_classes=(0,), fraud_requires_technique=False) -> pd.DataFrame:
    n = len(test_ids)
    for t, P in (("A", pA), ("B", pB), ("C", pC)):
        validate_probs(t, P, n, "submission input")
    label = np.asarray(pA).argmax(1).astype(int)
    action = np.asarray(pC).argmax(1).astype(int)
    thr = np.broadcast_to(np.asarray(technique_threshold, float), (len(TECHNIQUES),))
    techB = np.asarray(pB) >= thr
    if fraud_requires_technique:
        fraud_empty = ~np.isin(label, nonfraud_classes) & ~techB.any(1)
        techB[fraud_empty, np.asarray(pB)[fraud_empty].argmax(1)] = True
    techB[np.isin(label, nonfraud_classes)] = False
    conf = np.asarray(pA).max(1) if confidence is None else np.asarray(confidence, float)
    conf = np.clip(conf, 0.0, 1.0)
    return pd.DataFrame({
        "id": np.asarray(test_ids),
        "label": label,
        "techniques": [binary_to_techniques(r) for r in techB],
        "action": action,
        "confidence": conf.round(6),
    })


def validate_submission(sub: pd.DataFrame, sample: pd.DataFrame | None = None,
                        nonfraud_classes=(0,)) -> list[str]:
    """Returns a list of problems (empty list = valid)."""
    p = []
    if list(sub.columns) != SUBMISSION_COLS:
        return [f"columns must be exactly {SUBMISSION_COLS}, got {list(sub.columns)}"]
    if sub.isna().any().any():
        p.append(f"missing values in columns {sub.columns[sub.isna().any()].tolist()}")
    if sub["id"].duplicated().any():
        p.append(f"{int(sub['id'].duplicated().sum())} duplicate ids")
    for col in ("label", "action"):
        v = pd.to_numeric(sub[col], errors="coerce")
        if not pd.api.types.is_integer_dtype(sub[col]):
            p.append(f"{col} must be integer dtype, got {sub[col].dtype}")
        if v.isna().any() or not v.between(0, 5).all():
            p.append(f"{col} has values outside 0..5")
    bad_t = ~sub["techniques"].astype(str).str.match(_TECH_RE)
    if bad_t.any():
        p.append(f"{int(bad_t.sum())} malformed techniques values, e.g. {sub.loc[bad_t, 'techniques'].head(3).tolist()}")
    else:
        dup = sub["techniques"].astype(str).map(lambda s: len(s.split("|")) != len(set(s.split("|"))))
        if dup.any():
            p.append(f"{int(dup.sum())} techniques values repeat a code")
    nf = sub["label"].isin(nonfraud_classes) & (sub["techniques"] != NONE_TOKEN)
    if nf.any():
        p.append(f"{int(nf.sum())} non-fraud rows (label in {list(nonfraud_classes)}) have techniques != NONE")
    c = pd.to_numeric(sub["confidence"], errors="coerce")
    if c.isna().any() or not np.isfinite(c).all() or not c.between(0, 1).all():
        p.append("confidence must be finite and within [0, 1]")
    if sample is not None:
        if list(sample.columns) != SUBMISSION_COLS:
            p.append(f"sample_submission columns differ from expected: {list(sample.columns)}")
        if len(sub) != len(sample):
            p.append(f"row count {len(sub)} != sample_submission {len(sample)}")
        if set(sub["id"].tolist()) != set(sample["id"].tolist()):
            missing = len(set(sample["id"]) - set(sub["id"]))
            extra = len(set(sub["id"]) - set(sample["id"]))
            p.append(f"id set differs from sample_submission: {missing} missing, {extra} extra")
    return p


def write_submission(sub: pd.DataFrame, path, sample: pd.DataFrame | None = None) -> pd.DataFrame:
    """Order rows like sample_submission (if given), validate, then write. Raises on any problem."""
    if sample is not None and set(sub["id"].tolist()) == set(sample["id"].tolist()):
        sub = sample[["id"]].merge(sub, on="id", how="left", validate="one_to_one")
    problems = validate_submission(sub, sample)
    if problems:
        raise ValueError("submission invalid:\n  " + "\n  ".join(problems))
    sub.to_csv(path, index=False, lineterminator="\n")
    return sub

In [ ]:
%%writefile scamshield_src/augmentation/__init__.py
# package marker

In [ ]:
%%writefile scamshield_src/augmentation/corrupt.py
"""Deterministic adversarial text corruption, modelled on the noise types visible in the TRAINING data
(homoglyphs like 'sаlаry', zero-width spaces, injected junk tokens like 'xkcdpl', letter repeats like
'PIIIN', digit-for-letter swaps like 'P1N', typos, emoji injection).

Used by:
  - common/stress.py  -> robustness proxy (corrupt VALIDATION text only, never the training fold)
  - Samprity's adversarial augmentation (corrupt TRAINING-fold text only)
"""
from __future__ import annotations

import random
import string

_HOMO = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х", "i": "і", "y": "у"}
_LEET = {"i": "1", "o": "0", "e": "3", "a": "4", "s": "5", "l": "1"}
_EMOJI = ["🎁", "🔥", "💰", "✅", "⚠️", "👉"]
_ZW = ["​", "‌", "‍"]


def _junk(rng: random.Random) -> str:
    return "".join(rng.choice(string.ascii_lowercase) for _ in range(rng.randint(5, 10)))


def corrupt(text: str, seed: int, strength: float = 0.15) -> str:
    """Apply random character/token corruptions. Same (text, seed) -> same output."""
    rng = random.Random(f"{seed}:{text}")
    chars = list(text)
    out = []
    for ch in chars:
        low = ch.lower()
        r = rng.random()
        if ch.isalpha() and ch.isascii() and r < strength:
            op = rng.randint(0, 5)
            if op == 0 and low in _HOMO:
                out.append(_HOMO[low]); continue
            if op == 1 and low in _LEET:
                out.append(_LEET[low]); continue
            if op == 2:
                out.append(ch * rng.randint(2, 4)); continue
            if op == 3:
                continue  # drop char
            if op == 4:
                out.append(ch.swapcase()); continue
            out.append(ch + rng.choice(_ZW)); continue
        out.append(ch)
    s = "".join(out)
    toks = s.split(" ")
    if rng.random() < 0.6:
        toks.insert(rng.randint(0, len(toks)), _junk(rng))
    if rng.random() < 0.3:
        toks.insert(rng.randint(0, len(toks)), rng.choice(_EMOJI))
    return " ".join(toks)

In [ ]:
%%writefile scamshield_src/models/__init__.py
# package marker

In [ ]:
%%writefile scamshield_src/models/classical/__init__.py
# package marker

In [ ]:
%%writefile scamshield_src/models/classical/tfidf_lr.py
"""Classical baseline: char_wb + word TF-IDF -> Logistic Regression for Tasks A, B (6 binary), C.

CPU only, deterministic. Vectorizers are fit inside each training fold (no leakage).
Handles classes / techniques that are absent from a training fold:
  - multiclass: missing classes get probability 0, then rows are renormalized
  - binary technique with a single class in the fold: constant probability = fold prevalence
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from common.labels import N_A, N_B, N_C, TECHNIQUES


@dataclass
class TfidfLRConfig:
    text_view: str = "normalized_text"
    char_ngram: tuple = (2, 5)
    word_ngram: tuple = (1, 2)
    char_max_features: int = 200_000
    word_max_features: int = 50_000
    C: float = 4.0
    class_weight: str | None = "balanced"
    max_iter: int = 2000
    seed: int = 42
    predict_chunk: int = 50_000


class TfidfLR:
    name = "tfidf_lr"

    def __init__(self, cfg: TfidfLRConfig | None = None):
        self.cfg = cfg or TfidfLRConfig()

    # ---- features ----
    def _fit_vec(self, texts):
        c = self.cfg
        self.vc = TfidfVectorizer(analyzer="char_wb", ngram_range=tuple(c.char_ngram), sublinear_tf=True,
                                  min_df=1, max_features=c.char_max_features, dtype=np.float32)
        self.vw = TfidfVectorizer(analyzer="word", ngram_range=tuple(c.word_ngram), sublinear_tf=True,
                                  min_df=1, max_features=c.word_max_features, token_pattern=r"(?u)\b\w+\b",
                                  dtype=np.float32)
        return hstack([self.vc.fit_transform(texts), self.vw.fit_transform(texts)]).tocsr()

    def _vec(self, texts):
        return hstack([self.vc.transform(texts), self.vw.transform(texts)]).tocsr()

    def _lr(self):
        c = self.cfg
        return LogisticRegression(C=c.C, class_weight=c.class_weight, max_iter=c.max_iter, random_state=c.seed)

    # ---- training ----
    def fit(self, df: pd.DataFrame):
        X = self._fit_vec(df[self.cfg.text_view].tolist())
        self.mA = self._lr().fit(X, df["label"].to_numpy())
        self.mC = self._lr().fit(X, df["action"].to_numpy())
        self.mB = []
        for t in TECHNIQUES:
            y = df[t].to_numpy()
            self.mB.append(self._lr().fit(X, y) if len(np.unique(y)) == 2 else float(y.mean()))
        return self

    @staticmethod
    def _full_proba(model, X, n_classes):
        P = np.zeros((X.shape[0], n_classes))
        P[:, model.classes_.astype(int)] = model.predict_proba(X)
        return P / P.sum(1, keepdims=True)

    def _predict_X(self, X) -> dict:
        B = np.column_stack([m.predict_proba(X)[:, 1] if not isinstance(m, float) else np.full(X.shape[0], m)
                             for m in self.mB])
        return {"A": self._full_proba(self.mA, X, N_A), "B": B, "C": self._full_proba(self.mC, X, N_C)}

    def predict(self, df: pd.DataFrame) -> dict:
        texts = df[self.cfg.text_view].tolist()
        parts = [self._predict_X(self._vec(texts[i:i + self.cfg.predict_chunk]))
                 for i in range(0, len(texts), self.cfg.predict_chunk)]
        if not parts:
            return {"A": np.zeros((0, N_A)), "B": np.zeros((0, N_B)), "C": np.zeros((0, N_C))}
        return {k: np.vstack([p[k] for p in parts]) for k in ("A", "B", "C")}

    def config_dict(self) -> dict:
        return {"model": self.name, **{k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(self.cfg).items()}}


def make_fit_predict(cfg: TfidfLRConfig | None = None):
    """Adapter for common.stress.run_cv / run_stress."""
    def fit_predict(train_df, valid_df):
        return TfidfLR(cfg).fit(train_df).predict(valid_df)
    return fit_predict

In [ ]:
import os, sys, json, platform
SRC = os.path.abspath('scamshield_src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('.')
os.environ['SCAMSHIELD_ARTIFACTS_DIR'] = os.path.join(WORK, 'artifacts')
os.environ['SCAMSHIELD_OUTPUTS_DIR'] = WORK
# Data dir: auto-detected under /kaggle/input (folder containing sample_submission.csv),
# or set SCAMSHIELD_DATA_DIR yourself when running elsewhere.
import numpy, pandas, sklearn, scipy, yaml
print('python', platform.python_version(), '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__, '| PyYAML', yaml.__version__)
from common.seed import set_seed, device_info
SEED = set_seed(42)
print('seed', SEED, device_info())

## Data (aggregate checks only)

In [ ]:
from common.data_loader import load_config, load_train, load_test, load_sample_submission, resolve_data_dir
cfg = load_config(os.path.join(SRC, 'configs', 'base.yaml'))
print('data dir:', resolve_data_dir(cfg))
train = load_train(cfg)
test = load_test(cfg)
sample = load_sample_submission(cfg)
print('train', train.shape, 'fingerprint', train.attrs['fingerprint'])
print('test', test.shape, '| text column:', test.attrs['text_col'], '| extra columns:', test.attrs['extra_columns'])
assert set(test['id']) == set(sample['id']), 'test ids differ from sample_submission'
# Aggregate statistics only — no individual test rows are displayed.
print('train Task A counts:', train['label'].value_counts().sort_index().to_dict())
print('test script share:', test['script'].value_counts(normalize=True).round(4).to_dict())
print('test conversation share:', round(float(test['is_conversation'].mean()), 4))

## Shared grouped folds (regenerated, fingerprint-checked)

In [ ]:
from pathlib import Path
from common import folds as F
os.environ['SCAMSHIELD_FOLDS_DIR'] = os.path.join(WORK, 'folds')
meta = F.make_all_folds(train, cfg)
EXPECTED_FOLD_FINGERPRINT = 'c357ab8ee361835a'
print('fold fingerprint', meta['main']['fingerprint'], '| expected', EXPECTED_FOLD_FINGERPRINT)
assert meta['main']['fingerprint'] == EXPECTED_FOLD_FINGERPRINT, 'fold manifest differs from the one used in development'

## Model: TF-IDF + logistic regression — grouped CV and local stress tests

In [ ]:
from common.stress import run_stress
from common.metrics import composite_proxy
from models.classical.tfidf_lr import TfidfLR, TfidfLRConfig, make_fit_predict
mcfg = TfidfLRConfig(seed=SEED)
report, oof = run_stress(make_fit_predict(mcfg), train, seed=SEED, repeats=True)
m = report['main']
comp = composite_proxy({'F_A': m['F_A'], 'F_B': m['F_B'], 'F_C': m['F_C'], 'Calibration': m['Calibration'],
                        'F_robust': report['F_robust_proxy'], 'F_novel': report['F_novel_proxy'],
                        'F_domain': report['F_domain_proxy']})
print('OOF (grouped CV):', {k: m[k] for k in ('F_A', 'F_B', 'F_C', 'Brier', 'Calibration')})
print('Local proxies (NOT Kaggle scores):', {k: report[k] for k in ('F_robust_proxy', 'F_novel_proxy', 'F_domain_proxy')})
print('Repeated grouped CV F_A:', report['repeated_F_A'])
print('LOCAL composite proxy:', comp['composite_proxy'])

## Final fit and test prediction

In [ ]:
from common.submission import build_submission, write_submission, validate_submission
model = TfidfLR(mcfg).fit(train)          # final model: all labeled rows
pred = model.predict(test)
sc = cfg['submission']
sub = build_submission(test['id'].to_numpy(), pred['A'], pred['B'], pred['C'],
                       technique_threshold=sc['technique_threshold'], nonfraud_classes=sc['nonfraud_classes'],
                       fraud_requires_technique=sc['fraud_requires_technique'])
sub = write_submission(sub, os.path.join(WORK, 'submission.csv'), sample)
print('submission.csv written:', sub.shape)
print('predicted label share:', sub['label'].value_counts(normalize=True).sort_index().round(4).to_dict())
print('validation problems:', validate_submission(sub, sample, sc['nonfraud_classes']))

## Error analysis

In [ ]:
import pandas as pd
from common.labels import TASK_A_NAMES
print('Per-class Task A (OOF):')
print(pd.DataFrame(m['per_class_A']).T)
print('Confusion matrix Task A (rows = true, cols = predicted):')
print(pd.DataFrame(m['confusion_A'], index=list(TASK_A_NAMES.values()), columns=range(6)))
print('Per-technique Task B (OOF):')
print(pd.DataFrame(m['per_technique_B']).T)
print('Slices (script / conversation):')
print(pd.DataFrame(report['slices_A']).T)

## Generalization analysis
- **Languages/scripts:** see the slice table. The 86 labeled rows contain only 6 Bangla-script rows; Latin-script
  rows mix English and Banglish (script detection cannot separate them).
- **Unseen brands:** `F_domain_proxy` holds out every row of a brand together.
- **Noise:** `F_robust_proxy` corrupts validation text (homoglyphs, zero-width characters, junk tokens, repeats).
- **Class confusions:** see the confusion matrix (most errors are scams predicted as Legitimate).
- These are local proxies; the hidden evaluation subsets are not available to us.